In [1063]:
from sage.all import *
import sage.libs.lrcalc.lrcalc as lrcalc
import math


In [1064]:
def degree(partition):
    return sum(partition)

def compare_by_degree(p1, p2):
    d1, d2 = degree(p1), degree(p2)
    if d1 < d2:
        return -1
    elif d1 > d2:
        return 1
    return 0


In [1065]:
degree((3,))

3

In [1066]:

def generate_partitions(n, max_part=None):
    """Generate all partitions of n."""
    if n == 0:
        yield ()
    else:
        if max_part is None or max_part > n:
            max_part = n
        for first in range(max_part, 0, -1):
            for rest in generate_partitions(n - first, first):
                yield (first,) + rest

p = (3, 1) 
n=5
parts = list(generate_partitions(5))
print(parts)




[(5,), (4, 1), (3, 2), (3, 1, 1), (2, 2, 1), (2, 1, 1, 1), (1, 1, 1, 1, 1)]


In [1067]:
def subset_partitions(partition):
    """Return all partitions with degree <= degree(partition)."""
    d = degree(partition)
    result = []
    for n in range(d + 1):
        result.extend(generate_partitions(n))
    return result


subset_partitions((3,1))

[(),
 (1,),
 (2,),
 (1, 1),
 (3,),
 (2, 1),
 (1, 1, 1),
 (4,),
 (3, 1),
 (2, 2),
 (2, 1, 1),
 (1, 1, 1, 1)]

In [1068]:

print("Degree of p:", degree(p))

q = (1, 1)
print("Compare p and q:", compare_by_degree(p, q))  # 0 means equal

subset = subset_partitions(p)
print(f"All partitions with degree ≤ {degree(p)}:")
print(subset)


Degree of p: 4
Compare p and q: 1
All partitions with degree ≤ 4:
[(), (1,), (2,), (1, 1), (3,), (2, 1), (1, 1, 1), (4,), (3, 1), (2, 2), (2, 1, 1), (1, 1, 1, 1)]


# Calculating 2 LW Coefficient Manually Using COmbinatorics

In [1069]:

from itertools import permutations

def is_partition(p):
    return all(p[i] >= p[i+1] for i in range(len(p)-1))

def subtract_partitions(lam, mu):
    """Return skew shape lam/mu as list of row lengths."""
    if len(mu) > len(lam) or any(mu[i] > lam[i] for i in range(len(mu))):
        return None
    skew = [lam[i] - (mu[i] if i < len(mu) else 0) for i in range(len(lam))]
    return skew

def weight_to_list(weight):
    """Expand weight partition to list, e.g. (2,1) -> [1,1,2]."""
    res = []
    for i, m in enumerate(weight):
        res += [i+1]*m
    return res

def is_yamanouchi(word):
    """Check lattice word (Yamanouchi) condition."""
    counts = {}
    for x in word:
        counts[x] = counts.get(x, 0) + 1
        for y in range(1, x):
            if counts.get(y, 0) < counts[x]:
                return False
    return True

def lrcoefP(mu, nu, lam):
    """Compute c^{lam}_{mu,nu} using Yamanouchi condition."""
    skew = subtract_partitions(lam, mu)
    if skew is None:
        return 0
    num_boxes = sum(skew)
    if num_boxes != sum(nu):
        return 0

    # naive enumeration for small cases
    entries = weight_to_list(nu)
    c = 0
    for perm in set(permutations(entries)):
        if is_yamanouchi(perm):
            c += 1
    return c

In [1070]:
# Tests LW 2
print(lrcoefP((1,), (1,), (2,)))    # Expect 1  (since s_1 * s_1 = s_2 + s_{1,1})
print(lrcoefP((1,), (1,), (1,1)))   # Expect 1
print(lrcoefP((2,), (1,), (3,)))    # Expect 1
print(lrcoefP((2,), (1,), (2,1)))   # Expect 1
print(lrcoefP((2,), (2,), (4,)))    # Expect 1

1
1
1
1
1


In [1071]:
def solveOne(k, k1, k2, k3, k4):
    """
    Solve the system

        2*x2 + x1 + x3 + x4 = k
        x1 + x2 + x3 = k1 - k2
        x1 + x2 + x4 = k3 - k4

    for all non-negative integers x1, x2, x3, x4.

    Parameters
    ----------
    k  : int   – the total degree (soc^k)
    k1 : int   – |λ|
    k2 : int   – |λ'|
    k3 : int   – |μ|
    k4 : int   – |μ'|

    Returns
    -------
    List of dicts with all solutions:
        {"deg δ": x1, "deg γ": x2, "p": x3, "q": x4,
         "deg λ'": k2, "deg μ'": k4}
    """
    # sanity checks
    if k2 > k1 or k4 > k3:
        return []

    solutions = []

    # x1 can range from 0 up to k (reasonable upper bound)
    # !!!!!
    for x1 in range(k):
        # x1 + x2 + x3 = k1 - k2  => x3 = k1 - k2 - (x1 + x2)
        # x1 + x2 + x4 = k3 - k4  => x4 = k3 - k4 - (x1 + x2)
        # 2*x2 + x1 + x3 + x4 = k => 2*x2 + x1 + (k1 - k2 - x1 - x2) + (k3 - k4 - x1 - x2) = k
        # Simplify: (k1 - k2) + (k3 - k4) - x2 - x1 = k => x1 + x2 = (k1 - k2) + (k3 - k4) - k

        max_x2 = math.floor(((k1 - k2) + (k3 - k4) - (k-1) - x1)/2)
        if max_x2 < 0:
            continue

        for x2 in range(max_x2 + 1):
            x3 = k1 - k2 - (x1 + x2)
            x4 = k3 - k4 - (x1 + x2)

            if x3 < 0 or x4 < 0:
                continue

            solutions.append({
                "deg δ": x1,
                "deg γ": x2,
                "p": x3,
                "q": x4,
                "deg λ'": k2,
                "deg μ'": k4
            })

    return solutions


In [1072]:
# Examples: Correct Checked by Hand

print(solveOne(1, 1, 0, 1, 1))
print(solveOne(1, 1, 1, 1, 0))

# Ex1 by Professor

print(solveOne(1, 1, 0, 1, 0))






[{'deg δ': 0, 'deg γ': 0, 'p': 1, 'q': 0, "deg λ'": 0, "deg μ'": 1}]
[{'deg δ': 0, 'deg γ': 0, 'p': 0, 'q': 1, "deg λ'": 1, "deg μ'": 0}]
[{'deg δ': 0, 'deg γ': 0, 'p': 1, 'q': 1, "deg λ'": 0, "deg μ'": 0}, {'deg δ': 0, 'deg γ': 1, 'p': 0, 'q': 0, "deg λ'": 0, "deg μ'": 0}]


In [1073]:
def solveTwo(mu1, mu2, mu3, mu4):

    v1 = mu1 + mu2
    v2 = mu3 + mu4
    

    return [v1, v2]


In [1074]:
def restrict_partitions(part1_list, part2_list, d_lam):
    """
    Filters two lists of partitions (part1_list, part2_list) so that
    only pairs (a, b) satisfy |a| + |b| = |lam|, where |.| is the degree (sum of parts).

    Returns:
        (filtered_part1, filtered_part2)
        where both are lists of partitions that can potentially combine to lam.
    """

    filtered_part1 = []
    filtered_part2 = []

    for a in part1_list:
        for b in part2_list:
            if sum(a) + sum(b) == d_lam:
                filtered_part1.append(a)
                filtered_part2.append(b)

    return filtered_part1, filtered_part2


In [1075]:
# -----------------------------------------------------------------
#  three‑fold LR convolution – unchanged logic, corrected arguments
# -----------------------------------------------------------------
def lrcoef4(mu1, mu2, mu3, mu4, lam):
    """
    Return
        Σ_{a ⊢ (mu1+mu2)} Σ_{b ⊢ (mu3+mu4)}
               c^{a}_{mu1,mu2} · c^{b}_{mu3,mu4} · c^{lam}_{a,b}

    The first four arguments are *integers* (the degrees that appear in the
    linear system).  ``lam`` must be the *partition* λ (a tuple), not its
    total degree.
    """

    d_mu1 = degree(mu1)
    d_mu2 = degree(mu2)
    d_mu3 = degree(mu3)
    d_mu4 = degree(mu4)
    d_lam = degree(lam)
    
    total1 = d_mu1 + d_mu2                # degree of the first intermediate partition
    total2 = d_mu3 + d_mu4                # degree of the second intermediate partition

    print(f'deg of v1:{total1}')
    print(f'deg of v2:{total2}')
    parts_a = list(generate_partitions(total1))
    parts_b = list(generate_partitions(total2))

    parts_v1, parts_v2 = restrict_partitions(parts_a, parts_b, d_lam)


    # ----- DEBUG -------
    print("sum mu:", mu1+mu2+mu3+mu4, "sum lam:", sum(lam))

    print(f'possible partitions for v1: {parts_a}\n')
    print(f'possible partitions for v2: {parts_b}\n')
    print(type(parts_a[0]), parts_a[0])



    s = 0
    for a in parts_a:
        c1 = int(lrcoefP(mu1, mu2, a))   # c^{a}_{mu1,mu2}
        print(f'c1 = {c1}')
        if c1 == 0:
            continue
        for b in parts_b:
            c2 = int(lrcoefP(mu3, mu4, b))   # c^{b}_{mu3,mu4}
            print(f'c2 = {c2}')
            if c2 == 0:
                continue
            c3 = int(lrcoefP(a, b, lam))           # c^{lam}_{a,b}
            print(f'c3 = {c3}')
            if c3 == 0:
                continue
            s += c1 * c2 * c3
    return s


In [1076]:
print(f'{lrcoef4((1,), (), (), (1,), (1,1))} \n done! \n')
print(f'{lrcoef4((), (), (), (1,), (1,))} \n done! \n')
print(f'{lrcoef4((1,), (), (1,), (), (1,1))}')








deg of v1:1
deg of v2:1
sum mu: (1, 1) sum lam: 2
possible partitions for v1: [(1,)]

possible partitions for v2: [(1,)]

<class 'tuple'> (1,)
c1 = 1
c2 = 1
c3 = 1
1 
 done! 

deg of v1:0
deg of v2:1
sum mu: (1,) sum lam: 1
possible partitions for v1: [()]

possible partitions for v2: [(1,)]

<class 'tuple'> ()
c1 = 1
c2 = 1
c3 = 1
1 
 done! 

deg of v1:1
deg of v2:1
sum mu: (1, 1) sum lam: 2
possible partitions for v1: [(1,)]

possible partitions for v2: [(1,)]

<class 'tuple'> (1,)
c1 = 1
c2 = 1
c3 = 1
1


In [1077]:
def ones_partition(x):
    """Return the partition (1, 1, ..., 1) with x ones."""
    if x < 0:
        raise ValueError("x must be nonnegative")
    return tuple(1 for _ in range(x))


In [1078]:
def CalcSoc(k, lam, lamP, mu, muP):
    """
    k1 = |lam|,   k3 = |mu|
    """

    print(f'k value: {k} \n')
    k1 = degree(lam)
    k2 = degree(lamP)
    k3 = degree(mu)
    k4 = degree(muP)

    print (f'Values of deg lam, deg lamP, deg mu, deg muP, {k1, k2, k3, k4} \n')
    
    L = solveOne(k, k1, k2, k3, k4)

    print(L)

    tot_sum = 0
    for sol in L:
        d_list = list(generate_partitions(sol["deg δ"]))
        g_list = list(generate_partitions(sol["deg γ"]))
        p = ones_partition(sol["p"])
        q = ones_partition(sol["q"])

        for d in d_list:
            for g in g_list:
                print("\n=========== START OF EACH PRODUCT ========\n")
                print(f'solution partitions: d:{d}, g: {g}, 1^p:{p}, 1^q{q}\n')


                # first factor uses the *full* partition lam
                term1 = lrcoef4(lamP, d, g, p, lam)
                print(f'term1 in product (upper is lam) {term1}')

                # second factor uses the *full* partition mu
                term2 = lrcoef4(muP, d, g, q, mu)
                print(f'term2 in product (upper is mu)  {term2}')

                tot_sum += term1 * term2
    return tot_sum


In [1079]:
def Master1(k, lam, lamP, mu, muP, ex_string):
    print(f'\n--------{ex_string}-------- \n')
    print(solveOne(k, degree(lam), degree(lamP), degree(mu), degree(muP)))
    print(CalcSoc(k, lam, lamP, mu, muP))
    print("\n --------------------Done!-------------------- !!\n")


# My Example with Master

# Master1(3, (1, 2), (1, 1), (2,), (), 'My Example')
# My Example

# Ex 1

# Master1(1, (2,), (2,), (1,), (1,), 'Example 1')


# Ex 2

# Master1(3, (1,), (), (1,), (), 'Example 2')


# Ex 3

# Master1(2, (1,), (), (1,), (), 'Example 3')

# Ex 4

Master1(3, (1,1), (), (1,), (), 'Example 4')

# Ex 5

Master1(2, (1,1), (1,), (1,), (), 'Example 5') # CHECKS FINE!!!


# new example

# Master1(2, (1,), (), (1,), (1,), "new example")




--------Example 4-------- 

[{'deg δ': 0, 'deg γ': 0, 'p': 2, 'q': 1, "deg λ'": 0, "deg μ'": 0}, {'deg δ': 1, 'deg γ': 0, 'p': 1, 'q': 0, "deg λ'": 0, "deg μ'": 0}]
k value: 3 

Values of deg lam, deg lamP, deg mu, deg muP, (2, 0, 1, 0) 

[{'deg δ': 0, 'deg γ': 0, 'p': 2, 'q': 1, "deg λ'": 0, "deg μ'": 0}, {'deg δ': 1, 'deg γ': 0, 'p': 1, 'q': 0, "deg λ'": 0, "deg μ'": 0}]

=========== START OF EACH PRODUCT ========

solution partitions: d:(), g: (), 1^p:(1, 1), 1^q(1,)

deg of v1:0
deg of v2:2
sum mu: (1, 1) sum lam: 2
possible partitions for v1: [()]

possible partitions for v2: [(2,), (1, 1)]

<class 'tuple'> ()
c1 = 1
c2 = 1
c3 = 1
c2 = 1
c3 = 1
term1 in product (upper is lam) 2
deg of v1:0
deg of v2:1
sum mu: (1,) sum lam: 1
possible partitions for v1: [()]

possible partitions for v2: [(1,)]

<class 'tuple'> ()
c1 = 1
c2 = 1
c3 = 1
term2 in product (upper is mu)  1

=========== START OF EACH PRODUCT ========

solution partitions: d:(1,), g: (), 1^p:(1,), 1^q()

deg of v1:1
deg o